# CW2 starter pipeline — predicting `HighHealthBurden`

This notebook is your starting point, **not** a solution. It runs end to end and it
produces a number, which is exactly what makes it dangerous: a pipeline that runs
without error can still be a bad pipeline.

**What it does**

1. Loads the development dataset
2. Handles data quality issues in the crudest way available
3. Encodes non-numeric columns to numbers
4. Fits two models at their default settings
5. Evaluates on a single hold-out split, using accuracy
6. Scores the held-out test set

**What it deliberately does not do** — this is your assignment:

- it does not scale feature values
- it does not address the class imbalance
- it does not tune any hyperparameter
- it uses a single hold-out split rather than repeated splits or cross-validation
- it reports accuracy as though accuracy were the point
- it does not question whether every column *should* be a feature

Your job is to work out which of these matter, by how much, and for whom.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Both stakeholders' cost matrices, and helpers to apply them.
# See STAKEHOLDERS_AND_COSTS.md for where the numbers come from.
from costs import COST_MATRICES, cost_per_person, confusion_counts

RANDOM_STATE = 42
TARGET = 'HighHealthBurden'

## 2. Load the development data

`HighHealthBurden` is the target: did this person go on to have a high-burden health
year — fourteen or more days of health-related absence, or six or more primary-care
contacts — in the twelve months after the survey?

In [2]:
dev = pd.read_csv('coffee_health_v3_dev.csv')
print(f'{len(dev)} rows, {dev.shape[1]} columns')
print(f'\nPositive rate: {dev[TARGET].mean():.1%}')
dev.head()

10040 rows, 18 columns

Positive rate: 19.8%


,ID,Country,Age,Gender,Household Income,Smoking Status,Alcohol Level,Daily Coffees,Caffeine Intake,Stress Level,Physical Activity Level,BMI,Avg Resting Heart Rate,Avg Sleep Hours Per Night,Sleep Quality,Health Issues,SelfRatedHealth,HighHealthBurden
0,1,Norway,44.0,Male,36100.0,Light Smoker,Heavy,4.0,495.2,Medium,Moderately Active,24.7,NaN,4.1,Poor,Mild,Fair,1
1,2,Norway,32.0,Female,74200.0,Light Smoker,Light,1.7,168.5,Medium,Very Active,18.9,81.0,NaN,Good,No Issues,Very Good,0
2,3,Italy,47.0,Female,44400.0,Never,Moderate,2.9,196.7,Low,Lightly Active,25.1,83.0,NaN,Excellent,No Issues,Good,0
3,4,Norway,28.0,Female,103300.0,Never,Light,3.6,356.0,Medium,Very Active,27.5,70.0,7.6,Good,No Issues,Good,0
4,5,France,54.0,Female,53000.0,Never,Moderate,1.0,74.3,NaN,Moderately Active,26.4,71.0,8.1,Good,No Issues,Very Good,0


## 3. Data quality — the crude version

Duplicates get dropped, and any row with a missing value anywhere gets dropped with
it. This is the least effort possible. Note how many rows it costs you.

In [3]:
before = len(dev)
data = dev.drop_duplicates()
print(f'dropped {before - len(data)} duplicate rows')

before = len(data)
data = data.dropna()
print(f'dropped {before - len(data)} rows containing a missing value')
print(f'{len(data)} rows remain out of {len(dev)} ({len(data)/len(dev):.0%})')

dropped 40 duplicate rows
dropped 2615 rows containing a missing value
7385 rows remain out of 10040 (74%)


## 4. Encoding

Models need numbers. Ordered categories become integers on their natural scale;
unordered ones become dummy columns.

Everything that is not the target is used as a feature.

In [4]:
ORDINAL_SCALES = {
    'Stress Level': ['Low', 'Medium', 'High'],
    'Physical Activity Level': ['Sedentary', 'Lightly Active', 'Moderately Active', 'Very Active'],
    'Sleep Quality': ['Poor', 'Fair', 'Good', 'Excellent'],
    'Health Issues': ['No Issues', 'Mild', 'Moderate', 'Severe'],
    'Smoking Status': ['Never', 'Former', 'Vaper', 'Light Smoker', 'Heavy Smoker'],
    'Alcohol Level': ['Non-Drinker', 'Light', 'Moderate', 'Heavy'],
    'SelfRatedHealth': ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
}

def encode(df):
    d = df.copy()
    for column, order in ORDINAL_SCALES.items():
        d[column] = d[column].map({value: i for i, value in enumerate(order)})
    d = pd.get_dummies(d, columns=['Country', 'Gender'], drop_first=True)
    d = d.drop(columns=['ID'])            # an identifier is not a feature
    return d

encoded = encode(data)
y = encoded.pop(TARGET)
X = encoded.astype(float)
print(f'{X.shape[1]} features:')
print(list(X.columns))

19 features:
['Age', 'Household Income', 'Smoking Status', 'Alcohol Level', 'Daily Coffees', 'Caffeine Intake', 'Stress Level', 'Physical Activity Level', 'BMI', 'Avg Resting Heart Rate', 'Avg Sleep Hours Per Night', 'Sleep Quality', 'Health Issues', 'SelfRatedHealth', 'Country_Italy', 'Country_Norway', 'Country_UK', 'Gender_Male', 'Gender_Other']


## 5. Train / test split

One split, one seed.

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'train {len(X_train)} rows, validation {len(X_val)} rows')

train 5908 rows, validation 1477 rows


## 6. Two models, at their default settings

No parameters are chosen, tuned or justified.

In [6]:
models = {
    'Logistic Regression': LogisticRegression(),
    'K-Nearest Neighbours': KNeighborsClassifier(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'fitted {name}')

fitted Logistic Regression
fitted K-Nearest Neighbours


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 7. Evaluation

Accuracy, and then the cost each stakeholder would actually incur.

The cost matrices come from `costs.py`. A **negative** cost is a net benefit — a
correct invitation saves more than the programme costs to run.

In [7]:
def evaluate(name, model, X_eval, y_eval):
    predictions = model.predict(X_eval)
    counts = confusion_counts(y_eval, predictions)
    print(f'\n{name}')
    print(f'  accuracy: {accuracy_score(y_eval, predictions):.3f}')
    print(f'  confusion: TN={counts["TN"]}  FP={counts["FP"]}  '
          f'FN={counts["FN"]}  TP={counts["TP"]}')
    for stakeholder, matrix in COST_MATRICES.items():
        print(f'  cost to {stakeholder}: '
              f'EUR {cost_per_person(y_eval, predictions, matrix):+.2f} per person')

for name, model in models.items():
    evaluate(name, model, X_val, y_val)


Logistic Regression
  accuracy: 0.797
  confusion: TN=1173  FP=11  FN=289  TP=4
  cost to Public health agency: EUR +283.11 per person
  cost to Employer occupational health: EUR +156.28 per person



K-Nearest Neighbours
  accuracy: 0.766
  confusion: TN=1114  FP=70  FN=276  TP=17
  cost to Public health agency: EUR +271.20 per person
  cost to Employer occupational health: EUR +169.48 per person


## 8. The held-out test set

A separate, clean extract from a later recruitment wave. It is scored **once**, at the
end, and it is the common benchmark everyone's results are compared on.

The only things you may do to it are encoding and (once you add it) scaling. You must
not clean it, impute it, resample it, or tune anything against it.

In [8]:
test = pd.read_csv('coffee_health_v3_test.csv')

encoded_test = encode(test)
y_test = encoded_test.pop(TARGET)
X_test = encoded_test.astype(float).reindex(columns=X.columns, fill_value=0)

print(f'{len(test)} rows, positive rate {y_test.mean():.1%} '
      f'(development set: {y.mean():.1%})')

for name, model in models.items():
    evaluate(name, model, X_test, y_test)

3060 rows, positive rate 25.6% (development set: 19.5%)

Logistic Regression
  accuracy: 0.734
  confusion: TN=2198  FP=78  FN=736  TP=48
  cost to Public health agency: EUR +342.05 per person
  cost to Employer occupational health: EUR +199.61 per person

K-Nearest Neighbours
  accuracy: 0.712
  confusion: TN=2139  FP=137  FN=745  TP=39
  cost to Public health agency: EUR +351.91 per person
  cost to Employer occupational health: EUR +212.16 per person


## 9. Where you come in

You now have numbers. Before you try to improve them, work out what they mean.

- Compare the accuracy figures above against simply predicting "no" for everybody.
  What does that tell you about accuracy as a metric here?
- Look at the confusion matrices. How many of the people who actually had a
  high-burden year did each model find?
- The two stakeholders do not agree about which model is better. Why not?
- Test performance differs from validation performance. How much of that is your
  model, and how much is the test set?

Then start improving the pipeline, and **measure every change** — including the ones
that turn out not to help.